In [1]:
import pandas as pd
import numpy as np
import pickle, faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
movies=pickle.load(open('movies.pkl','rb')).reset_index(drop=True)
print("titles:",len(movies))

titles: 4806


In [3]:
emb=np.load('models/embeddings.npy')
emb

array([[-0.00839981,  0.00342847,  0.0078335 , ...,  0.00565999,
        -0.02905595, -0.03096052],
       [-0.0589229 , -0.08379569, -0.0312748 , ..., -0.06853222,
        -0.00240157, -0.05069666],
       [-0.14067364, -0.04145731, -0.02567357, ..., -0.06887827,
        -0.00265004, -0.01375366],
       ...,
       [-0.06737365, -0.04942525,  0.01014039, ..., -0.00952996,
         0.03990992, -0.03229108],
       [-0.07358247, -0.05269022,  0.01147517, ..., -0.02450187,
         0.01654166, -0.0176364 ],
       [-0.07923176, -0.0014256 ,  0.02443677, ..., -0.00949923,
         0.01516946,  0.04482699]], shape=(4806, 384), dtype=float32)

In [9]:
index=faiss.read_index('models/movie_index.faiss')
print("index type:",type(index).__name__)
print("ntotal:",index.ntotal)

index type: IndexFlatL2
ntotal: 4806


In [10]:
tfidf=TfidfVectorizer(max_features=5000, stop_words='english')
X=tfidf.fit_transform(movies['tags'])

In [11]:
def faiss_recs(i,k=5):
    _,idx=index.search(emb[i].reshape(1,-1),k+1)
    return [j for j in idx[0] if j!=i][:k]

In [12]:
def tfidf_recs(i,k=5):
    sims=cosine_similarity(X[i],X).ravel()
    sims[i]=-1
    return list(np.argsort(-sims)[:k])

In [13]:
test_titles=["The Dark Knight","Toy Story","Spider-Man","The Godfather","The Conjuring"]

In [14]:
for t in test_titles:
    m=movies[movies['title']==t]
    if m.empty:print("\nMISSING:",t); continue
    i=m.index[0]
    i=m.index[0]
    print(f"\n==={t}===")
    print(" Embeddings+FAISS :",[movies.title[j] for j in faiss_recs(i)])
    print(" TF-IDF baseline :",[movies.title[j] for j in tfidf_recs(i)])


===The Dark Knight===
 Embeddings+FAISS : ['The Dark Knight Rises', 'Batman', 'Batman v Superman: Dawn of Justice', 'Batman', 'Batman Returns']
 TF-IDF baseline : ['The Dark Knight Rises', 'Batman Returns', 'Batman Begins', 'Batman: The Dark Knight Returns, Part 2', 'Batman Forever']

===Toy Story===
 Embeddings+FAISS : ['Toy Story 3', 'Toy Story 2', 'The Rugrats Movie', 'Celeste & Jesse Forever', 'Robot & Frank']
 TF-IDF baseline : ['Toy Story 2', 'Toy Story 3', 'For Your Consideration', 'Small Soldiers', "Losin' It"]

===Spider-Man===
 Embeddings+FAISS : ['The Amazing Spider-Man 2', 'Spider-Man 3', 'Spider-Man 2', 'The Amazing Spider-Man', 'Stuart Little']
 TF-IDF baseline : ['Spider-Man 3', 'Spider-Man 2', 'Arachnophobia', 'The Amazing Spider-Man 2', 'The Amazing Spider-Man']

===The Godfather===
 Embeddings+FAISS : ['The Godfather: Part II', 'The Godfather: Part III', 'The Conformist', 'GoodFellas', "Things to Do in Denver When You're Dead"]
 TF-IDF baseline : ['The Godfather: Par

In [15]:
ov=[len(set(faiss_recs(i))&set(tfidf_recs(i))) for i in range(0,len(movies),20)]
print("Avg overlap (of 5) between methods on sample:",round(np.mean(ov),2))

Avg overlap (of 5) between methods on sample: 0.64
